# Loading data
In our exploratory analysis, we are first going to check to see how our JSON data is structured. Down below, we printed out the first 5 actors to determine if our code is running properly.

In [19]:
import json

PATH = "../data/raw/data/v1.2.0/layers/surface.json"

with open(PATH, "r") as f:
    data = json.load(f)

# prints out the first 5 actors
print(list(data.keys())[:5])


['ActorLinkDeleteControlActor', 'AddAutoPlacementCreator', 'AirWall', 'AirWallCaveMaster', 'AirWallCharacterOnly']


# Printing out Lynel Actor
Here, we are going to decipher the Lynels from the dataset. Based on the output, we can see that there are 3 types of Lynels: Junior, Middle, and Senior. For each type of Lynel, they have the same bow, but with a different type of sword or spear. For example, `Enemy_Lynel_Junior` has `Weapon_Bow_009`, but they could also have `Weapon_Lsword_112`, `Weapon_Spear_112`, and `Weapon_Shield_016`.

In [20]:
for actor in data.keys():
    if actor.startswith("Enemy_Lynel"):
        print(actor)

Enemy_Lynel_Junior : Weapon_Bow_009 : Weapon_Lsword_112
Enemy_Lynel_Junior : Weapon_Bow_009 : Weapon_Spear_112
Enemy_Lynel_Junior : Weapon_Bow_009 : Weapon_Sword_112 : Weapon_Shield_016
Enemy_Lynel_Middle : Weapon_Bow_026 : Weapon_Lsword_112
Enemy_Lynel_Middle : Weapon_Bow_026 : Weapon_Lsword_113
Enemy_Lynel_Middle : Weapon_Bow_026 : Weapon_Spear_113
Enemy_Lynel_Middle : Weapon_Bow_026 : Weapon_Sword_112 : Weapon_Shield_017
Enemy_Lynel_Middle : Weapon_Bow_026 : Weapon_Sword_113 : Weapon_Shield_017
Enemy_Lynel_Senior : Weapon_Bow_032 : Weapon_Lsword_124
Enemy_Lynel_Senior : Weapon_Bow_032 : Weapon_Spear_124
Enemy_Lynel_Senior : Weapon_Bow_032 : Weapon_Sword_113 : Weapon_Shield_018


# Looking into Lynel Actor 
Here, we are going to explore more data about Lynels. Based on the output, we can see that there are 3 different of keys: `name`, `icons`, and `locations`. We will explore even further with these keys to find out what data we need.

In [21]:
for actor in data.keys():
    if actor.startswith("Enemy_Lynel"):
        print(data[actor].keys())
        break

dict_keys(['name', 'icons', 'locations'])


We are going to count individual Lynel placements just to see the overall data set of the Lynels.

In [22]:
total_lynels = 0

for actor in data.keys():
    if actor.startswith("Enemy_Lynel"):
        total_lynels += len(data[actor]["locations"])

print(f"Total Lynels: {total_lynels}")

Total Lynels: 15


# Data Validating

### Un-optimized Way to Analyze Data
Now that we know the Lynel placements, we will extract their locations into one list. As an exercise, we're going to use a nested `for-loop` just to have a better understanding of how data is flattened in a non-optimal way.

In [23]:
lynel_records = []

for actor in data:
    if actor.startswith("Enemy_Lynel"):
        locations = data[actor]["locations"]

        for location in locations:
            record = {
                "actor": actor,
                "x": location["x"],
                "y": location["y"],
                "z": location["z"],
                "hash": location["hash"],
            }

            lynel_records.append(record)

print(f"Total records: {len(lynel_records)}")
print(lynel_records[:3])

Total records: 15
[{'actor': 'Enemy_Lynel_Junior : Weapon_Bow_009 : Weapon_Lsword_112', 'x': 2959.004, 'y': 3680.687, 'z': 27.39, 'hash': '0x6b549b34c4879128'}, {'actor': 'Enemy_Lynel_Junior : Weapon_Bow_009 : Weapon_Spear_112', 'x': 30, 'y': -3378, 'z': 94.785, 'hash': '0x1429e6cbe00082af'}, {'actor': 'Enemy_Lynel_Junior : Weapon_Bow_009 : Weapon_Spear_112', 'x': 1551.755, 'y': -2169.749, 'z': 217.199, 'hash': '0xbd56cfe619f862ae'}]


### DataFrames

Here, we are going to do the same thing, but with DataFrames.

In [24]:
import pandas as pd

lynel_df = pd.DataFrame(lynel_records)

# prints out first 5 records in the dataframe
display(lynel_df.head())

# prints out the shape of the dataframe. (# of rows, # of columns)
print(lynel_df.shape)

# prints out the data types of each column
print(lynel_df.dtypes)

,actor,x,y,z,hash
0,Enemy_Lynel_Junior : Weapon_Bow_009 : Weapon_L...,2959.004,3680.687,27.390,0x6b549b34c4879128
1,Enemy_Lynel_Junior : Weapon_Bow_009 : Weapon_S...,30.000,-3378.000,94.785,0x1429e6cbe00082af
2,Enemy_Lynel_Junior : Weapon_Bow_009 : Weapon_S...,1551.755,-2169.749,217.199,0xbd56cfe619f862ae
3,Enemy_Lynel_Junior : Weapon_Bow_009 : Weapon_S...,1839.559,-2799.320,308.423,0xcf9facf3b321a85a
4,Enemy_Lynel_Junior : Weapon_Bow_009 : Weapon_S...,138.469,-1553.582,109.097,0xbba5089292f5b303


(15, 5)
actor        str
x        float64
y        float64
z        float64
hash         str
dtype: object


Now, we are going to check to see if our dataset is reliable. We are going to check to see if there are any missing values or duplicate identifiers.

In [25]:
# is.na() checks to see if there are any missing values in the dataframe. sum() counts the number of missing values in each column.
print(lynel_df.isna().sum())

# we are checking to see if there are any duplicated hashes
print(lynel_df['hash'].duplicated().sum())

actor    0
x        0
y        0
z        0
hash     0
dtype: int64
0


# Feature Engineering
We are going to create new variables from existing data to make it more useful for analysis. First, we are going to create a column called "variant" and store different types of Lynels in that column. For practice, we are going to create a method called `extract_variant()` that extracts the actor name and returns the variant type.

In [26]:
def extract_variant(actor):
    variant_and_equipment =  actor.split("_")[2]
    variant = variant_and_equipment.split(" : ")
    return variant[0]


for actor in data.keys():
    if actor.startswith("Enemy_Lynel"):
        print(extract_variant(actor))

Junior
Junior
Junior
Middle
Middle
Middle
Middle
Middle
Senior
Senior
Senior


# Applying functions to a pandas Dataframe
We are goign to store the variants in the variants column.

In [ ]:
lynel_df['variants'] = lynel_df['actor'].apply(extract_variant)

# value_copunts() counts the number of occurrences of each unique value in the 'variants' column.
print(lynel_df['variants'].value_counts())

variants
Middle    7
Junior    5
Senior    3
Name: count, dtype: int64


# Data Visualization
Now that we explored the distribution numerically, we are going to create a visualization.

In [ ]:
import matplotlib.pyplot as plt

variant_counts = lynel_df['variants'].value_counts()

lynel_variant = variant_counts.index
num_of_lynels = variant_counts.values

plt.bar(lynel_variant, num_of_lynels)
plt.title("Frequency of Lynels by Variant")
plt.xlabel("Lynel Variant")
plt.ylabel("# of Lynels")
plt.show()


Index(['Middle', 'Junior', 'Senior'], dtype='str', name='variants')
[7 5 3]
